## Exploratory Data Analysis

In [7]:
import numpy as np
import pandas as pd
import yfinance as yf
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings("ignore")

In [11]:
import os
repo_url = "https://github.com/tongyuguo/HelpHerInvest.git"
repo_dir = "HelpHerInvest"
if not os.path.exists(repo_dir):
    !git clone {repo_url}
else:
    !git -C {repo_dir} pull

# list available data files
!ls -lah {repo_dir}/Data

# Create Data directory path 
DATA_DIR = Path(repo_dir) / "Data"



error: Pulling is not possible because you have unmerged files.
hint: Fix them up in the work tree, and then use 'git add/rm <file>'
hint: as appropriate to mark resolution and make a commit.
fatal: Exiting because of an unresolved conflict.
total 16M
drwxr-xr-x  2 jupyter-toomeyck jupyter-toomeyck  162 Feb  4 00:52 .
drwxr-xr-x 11 jupyter-toomeyck jupyter-toomeyck 4.0K Jan 31 00:24 ..
-rw-r--r--  1 jupyter-toomeyck jupyter-toomeyck 2.4M Feb  4 01:03 dependent_variables.csv
-rw-r--r--  1 jupyter-toomeyck jupyter-toomeyck 677K Feb  4 01:03 independent_variables.csv
-rw-r--r--  1 jupyter-toomeyck jupyter-toomeyck 2.9M Feb  4 01:03 production_dataset.csv
-rw-r--r--  1 jupyter-toomeyck jupyter-toomeyck 9.4M Jan 31 03:13 stock_symbols_new.csv.zip
-rw-r--r--  1 jupyter-toomeyck jupyter-toomeyck 296K Jan 30 21:45 testing_small.csv.zip


In [15]:
import zipfile

# Load the dataset
dataset_path2= DATA_DIR / "final_dataset.csv.zip"
print(dataset_path2)
csv_file_name = "final_dataset.csv"

dataset_path1 = f"{repo_dir}/Data/stock_symbols_new.csv.zip"
csv_file_name = "stock_symbols_new.csv"
csv_path = f"{repo_dir}/Data/{csv_file_name}"
print(dataset_path1)
with zipfile.ZipFile(dataset_path1, 'r') as zf:
    with zf.open(csv_file_name) as file_handle:
        # Read the file-like object directly into pandas
        symbols = pd.read_csv(file_handle)


with zipfile.ZipFile(dataset_path2, 'r') as zf:
    with zf.open(csv_file_name) as file_handle:
        # Read the file-like object directly into pandas
        symbols = pd.read_csv(file_handle)

HelpHerInvest/Data/final_dataset.csv.zip
HelpHerInvest/Data/stock_symbols_new.csv.zip


FileNotFoundError: [Errno 2] No such file or directory: 'HelpHerInvest/Data/final_dataset.csv.zip'

In [5]:
# Perform basic EDA
count_nas = merged_df.isna().sum()
print("Missing values per column:")
print(count_nas)

# Summary statistics
merged_df.describe()


AttributeError: 'PosixPath' object has no attribute 'isna'

We retrieved the summary statistics for the "fwd_excess" variable of the 50 tickers generated. Additionally, we drilled down to each ticker to and sorted them by the average "fwd_eccces" variable to see which tickers had the highest values. Many of the tickers are highly traded symbols now (PLTR, TSLA, NVDA, etc.), but one thing that jumps out is how PLTR doesn't have as many observations as the other tickers because it is a recently listed symbol.

In [ ]:
# Analyze the target variable 'fwd_excess'
print(merged_df['fwd_excess'].describe())

# By ticker - mean
ticker_group = merged_df.groupby("Ticker")['fwd_excess'].describe().sort_values('mean', ascending=False)
ticker_group

# By ticker - sum
#ticker_group = merged_df.groupby("Ticker")['fwd_excess'].sum().sort_values(ascending=False)
#ticker_group

Below is a chart that displays the average "fwd_excess" over the course of the review period. This determines the average across all 50 tickers and provides a good representation of what this portfolio of 50 names could potentially return against the S&P 500 if the weights were evenly distributed. This is something interesting to note.

In [ ]:
# Average 'fwd_excess' over time
date_group = merged_df.groupby('Date')['fwd_excess'].mean()
date_group.plot(title='Mean "fwd_excess" Over Time', figsize=(12,6), ylabel='Mean fwd_excess', xlabel='Date')

In [ ]:
# Histogram of variables

cols = merged_df.columns.drop(['fwd_excess','Date'])
merged_df[cols].hist(figsize=(14, 10), bins=50)
plt.tight_layout()
plt.show()

The histograms above display the distribution of values for each variable. Many of these are fairly normally distributed, but the volatility variables are slightly right-skewed and the drawdown variables are slightly left-skewed.

In [ ]:
# Correaltion Anlaysis

# select only numeric columns for correlation
numeric_data = merged_df.select_dtypes(include=[np.number])
corr = merged_df.select_dtypes(include='number').corr()

fwd_excess_corr = numeric_data.corrwith(numeric_data['fwd_excess']).sort_values()
print("Correlation of features with 'fwd_excess':")
print(fwd_excess_corr.tail(15))

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.2)
plt.title('Correlation Heatmap (All Numeric Columns)')
plt.tight_layout()
plt.show()


The correlation analysis does not provide strong positive or negative results for the "fwd_excess" variable. I am curious to see if this is consistent with most tickers and if the timeframe is too wide. Adjusting these parameters may produce stronger correlations. 

In [ ]:
# Example rolling correlation by date
tmp = merged_df.reset_index()
tmp = tmp.sort_values('Date')
roll = tmp.groupby('Date')[['mom_12m','fwd_excess']].mean().rolling(12).corr().iloc[0::2,1]
roll.plot(title='Rolling corr: mom_12m vs fwd_excess', figsize=(14,8), ylabel='Correlation', xlabel='Date')


In [ ]:
''' 
Cross ‑ sectional analysis per date
For each month, examine signal effectiveness cross‑sectionally
Rank features by how well they explain fwd_excess
'''

# Cross-sectional correlation per date
cs_corr = tmp.groupby('Date').apply(lambda g: g['mom_12m'].corr(g['fwd_excess']))
cs_corr.plot(title='Cross-sectional corr by Date',figsize=(12,6), ylabel='Correlation', xlabel='Date')

In [ ]:
'''
Potential Outlier influence
Extreme fwd_excess tails: which tickers/dates drive them?
Winsorization sensitivity checks
'''

q = merged_df['fwd_excess'].quantile([0.01, 0.99])
tail = merged_df[(merged_df['fwd_excess']<=q.iloc[0]) | (merged_df['fwd_excess']>=q.iloc[1])]
tail.reset_index().groupby('Ticker').size().sort_values(ascending=False).head(10)